In [ ]:
!pip install google-api-python-client

In [ ]:
import os
import pandas as pd
from googleapiclient.discovery import build

In [ ]:
api_key = 'API Key'
youtube = build('youtube', 'v3', developerKey=api_key)

In [ ]:
def get_videos_from_playlist(playlist_id):
    video_ids = []
    next_page_token = None

    while True:
        # Fetch the playlist items
        playlist_response = youtube.playlistItems().list(
            part='snippet',
            playlistId=playlist_id,
            maxResults=50,  # Maximum allowed per request
            pageToken=next_page_token
        ).execute()

        # Extract video IDs
        for item in playlist_response['items']:
            video_ids.append(item['snippet']['resourceId']['videoId'])

        # Check if there's another page
        next_page_token = playlist_response.get('nextPageToken')
        if not next_page_token:
            break

    return video_ids

# Function to get video statistics for a list of video IDs
def get_video_statistics(video_ids):
    video_data = []

    # Process video IDs in batches (max 50 per request)
    for i in range(0, len(video_ids), 50):
        batch_ids = video_ids[i:i + 50]
        stats_response = youtube.videos().list(
            part='statistics,snippet',
            id=','.join(batch_ids)
        ).execute()

        # Extract data for each video
        for item in stats_response['items']:
            video_data.append({
                'video_id': item['id'],
                'title': item['snippet']['title'],
                'views': item['statistics'].get('viewCount', 'N/A'),
                'likes': item['statistics'].get('likeCount', 'N/A'),
                'comments': item['statistics'].get('commentCount', 'N/A')
            })

    return video_data

# Playlist ID from the URL
playlist_id = 'PLRUDRdqa_lvfa1wY8YD35IYnNP0KUaLaf'

# Fetch video IDs from the playlist
video_ids = get_videos_from_playlist(playlist_id)

# Fetch video statistics for these video IDs
video_statistics = get_video_statistics(video_ids)

# Create a DataFrame
df = pd.DataFrame(video_statistics)

In [ ]:
playlist_old = 'PLA9abQ_oNFZlDO6ZeEgUXYd446foWbohA'
video_ids_old = get_videos_from_playlist(playlist_old)
video_statistics_old = get_video_statistics(video_ids_old)

In [ ]:
df_old = pd.DataFrame(video_statistics_old)

In [ ]:
young_thug_trial_videos = pd.concat([df_old, df], ignore_index=True)

In [ ]:
young_thug_trial_videos.to_csv('young_thug_trial_videos.csv', index=False)

In [ ]:
young_thug_trial_videos

,video_id,title,views,likes,comments
0,5IwZ266l564,"Young Thug, YSL trial stream Day 1",95047,847,187
1,BttFj68FpDQ,"Young Thug, YSL Trial Day 2 | Watch Live",67447,439,73
2,aYT4Q0vnzCc,"YSL, Young Thug trial Day 3 live stream",190461,933,257
3,ZOJR3kHE1gU,Young Thug YSL trial Day 4 live stream,81545,658,155
4,4fMcukrPv8E,YSL Young Thug trial Day 5 live stream,71886,462,134
...,...,...,...,...,...
171,kMilHBzJSdA,LIVE: YSL RICO Trial — GA v. Deamonte Kendrick...,125894,1209,73
172,Ug0dcWNcKu4,VERDICT WATCH: YSL RICO Trial — GA v. Deamonte...,187434,1330,64
173,QDo5FeqmdKI,VERDICT WATCH: YSL RICO Trial — GA v. Deamonte...,216831,1403,94
174,pV57qmDd5EI,VERDICT: YSL RICO Trial — GA v. Deamonte Kendr...,122463,1499,225


# Comment

In [ ]:
def get_video_comments(video_id):
    comments_data = []
    next_page_token = None

    while True:
        # Call the API to get comments
        response = youtube.commentThreads().list(
            part='snippet',
            videoId=video_id,
            maxResults=100,  # Maximum allowed per request
            pageToken=next_page_token
        ).execute()

        # Process each comment
        for item in response['items']:
            snippet = item['snippet']['topLevelComment']['snippet']
            comments_data.append({
                'video_id': video_id,
                'comment': snippet['textDisplay'],
                'likes': snippet.get('likeCount', 0),
                'replies': item['snippet'].get('totalReplyCount', 0)
            })

        # Check if there's another page of comments
        next_page_token = response.get('nextPageToken')
        if not next_page_token:
            break

    return comments_data

In [ ]:
all_comments = []
for video_id in young_thug_trial_videos['video_id']:
    comments = get_video_comments(video_id)
    all_comments.extend(comments)

# Convert the comments data into a DataFrame
comments_df = pd.DataFrame(all_comments)

In [ ]:
comments_df

,video_id,comment,likes,replies
0,5IwZ266l564,Her face is the definition of hate 😤😤😤,0,0
1,5IwZ266l564,First time watching from the beginning,2,0
2,5IwZ266l564,Love was always a lying B!!!!!!!!!!!!!!!!!! I ...,1,0
3,5IwZ266l564,My first time watching opening DA LOVE was sne...,2,0
4,5IwZ266l564,The strength of the wolf.. sit down love. Sh...,1,0
...,...,...,...,...
24690,zrbKIgY6Qcc,Why does the judge not let anyone speak then l...,203,10
24691,zrbKIgY6Qcc,No ya ain&#39;t!,3,0
24692,zrbKIgY6Qcc,This Judge is a disgrace,338,6
24693,zrbKIgY6Qcc,1st here,3,0


In [ ]:
comments_df.to_csv('young_thug_trial_comments.csv', index=False)

In [ ]:
youtube_young_thug_full_data = pd.merge(
    young_thug_trial_videos,  # Left DataFrame
    comments_df,              # Right DataFrame
    on='video_id',            # Column to join on
    how='outer'               # Full join
)

In [ ]:
youtube_young_thug_full_data.to_csv('youtube_young_thug_full_data.csv', index=False)